In [137]:
import cv2
import numpy as np
import math

# Importing and Preprocessing Any Map

In [138]:
map_ = cv2.imread("track3.jpg")
map_ = cv2.cvtColor(map_, cv2.COLOR_BGR2GRAY)

# Invert the image
_, thr = cv2.threshold(map_, 220, 255, cv2.THRESH_BINARY_INV)

# Find the largest connected area
num_labels, labels, stats, centroid = cv2.connectedComponentsWithStats(thr)
print("=====Number of Labels=====")
print(num_labels)
print("====Region Labels====")
print(labels)
print("=====Region Stats=====") #x,y,w,h,area
print(stats)
print("=====Centroid==========")
print(centroid)

cv2.imshow("path", thr)
cv2.waitKey()
cv2.destroyAllWindows()

=====Number of Labels=====
2
====Region Labels====
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]
=====Region Stats=====
[[    0     0   124   474 57124]
 [   14    54    94   402  1652]]
=====Centroid==========
[[ 61.08521812 235.58852671]
 [ 75.84261501 268.01755448]]


### Filtered Area

In [139]:
max_i = 0
max_area = 0

for i in range(1, num_labels):
    area = stats[i, cv2.CC_STAT_AREA]
    if area > max_area:
        max_i = i
        max_area = area

mask = (labels == max_i).astype(np.uint8)*255
filtered_map = cv2.bitwise_and(thr, thr, mask=mask)

cv2.imshow("test", filtered_map)
cv2.waitKey()
cv2.destroyAllWindows()

### Smoothing

In [140]:
# Find the skeleton
skeleton = cv2.ximgproc.thinning(filtered_map, thinningType=cv2.ximgproc.THINNING_ZHANGSUEN)

cv2.imshow("test", skeleton)
cv2.waitKey()
cv2.destroyAllWindows()

## Generate the Graph

!pip install sknw networkx

In [141]:
import sknw
import networkx as nx

graph = sknw.build_sknw((skeleton>0).astype(np.uint8), multi=False)

edges = list(graph.edges(data=True))
graph.edges

EdgeView([(0, 5), (0, 1), (1, 2), (2, 3), (3, 4), (4, 6), (5, 9), (6, 7), (7, 7), (7, 8), (8, 9)])

In [142]:
graph.nodes()

NodeView((0, 1, 2, 3, 4, 5, 6, 7, 8, 9))

In [143]:
graph.nodes[0]

{'pts': array([[58, 86],
        [59, 85],
        [59, 86],
        [60, 84],
        [60, 85],
        [61, 84],
        [62, 83],
        [62, 84],
        [63, 83]], dtype=int16),
 'o': array([60, 84], dtype=uint16)}

In [144]:
edges[0]

(0,
 5,
 {'pts': array([[ 60,  84],
         [ 57,  87],
         [ 56,  88],
         [ 55,  89],
         [ 55,  90],
         [ 54,  91],
         [ 54,  92],
         [ 55,  93],
         [ 55,  94],
         [ 56,  95],
         [ 57,  96],
         [ 58,  96],
         [ 59,  97],
         [ 60,  97],
         [ 61,  97],
         [ 62,  98],
         [ 63,  98],
         [ 64,  98],
         [ 65,  99],
         [ 66,  99],
         [ 67,  99],
         [ 68, 100],
         [ 69, 100],
         [ 70, 100],
         [ 71, 100],
         [ 72, 100],
         [ 73, 100],
         [ 74, 100],
         [ 75, 100],
         [ 76, 100],
         [ 77, 101],
         [ 78, 101],
         [ 79, 101],
         [ 80, 101],
         [ 81, 101],
         [ 82, 101],
         [ 83, 102],
         [ 84, 102],
         [ 85, 102],
         [ 86, 102],
         [ 87, 102],
         [ 88, 102],
         [ 89, 102],
         [ 90, 102],
         [ 91, 103],
         [ 92, 103],
         [ 93, 103]

In [145]:
import numpy as np

# Compute node coordinates from graph
node_coords = {n: np.array(graph.nodes[n]['o']) for n in graph.nodes()}
node_coords

{0: array([60, 84], dtype=uint16),
 1: array([67, 80], dtype=uint16),
 2: array([334,  70], dtype=uint16),
 3: array([342,  19], dtype=uint16),
 4: array([347,  14], dtype=uint16),
 5: array([417,  86], dtype=uint16),
 6: array([427,  24], dtype=uint16),
 7: array([432,  28], dtype=uint16),
 8: array([447,  42], dtype=uint16),
 9: array([448,  67], dtype=uint16)}

In [146]:
start_node = min(node_coords, key=lambda n: node_coords[n][0] + node_coords[n][1])
start_node

0

In [147]:
[n for n in graph.neighbors(0)]

[5, 1]

In [148]:
ordered_nodes = [start_node]
path = []
visited = {start_node}
cur = start_node

while True:
    neighbors = [n for n in graph.neighbors(cur) if n not in visited]
    if not neighbors:
        break
    next_node = neighbors[0]  # choose the first unvisited neighbor
    ordered_nodes.append(next_node)
    visited.add(next_node)
    path.append((cur, next_node))
    cur = next_node

path.append((cur, start_node))

In [149]:
path

[(0, 5),
 (5, 9),
 (9, 8),
 (8, 7),
 (7, 6),
 (6, 4),
 (4, 3),
 (3, 2),
 (2, 1),
 (1, 0)]

### Relabeling

In [150]:
mapping = {old:new for new, old in enumerate(ordered_nodes)}
graph = nx.relabel_nodes(graph, mapping, copy=True)

In [151]:
mapping

{0: 0, 5: 1, 9: 2, 8: 3, 7: 4, 6: 5, 4: 6, 3: 7, 2: 8, 1: 9}

In [152]:
ordered_nodes = [start_node]
path = []
visited = [start_node]
start_node = 0
cur = start_node

while True:
    neighbors = [n for n in graph.neighbors(cur) if n not in visited]
    if not neighbors:
        break
    visited.append(cur)
    next_node = neighbors[0]  # choose the first unvisited neighbor
    ordered_nodes.append(next_node)
    path.append((cur, next_node))
    cur = next_node

path.append((cur, 0))

In [153]:
path

[(0, 1),
 (1, 2),
 (2, 3),
 (3, 4),
 (4, 5),
 (5, 6),
 (6, 7),
 (7, 8),
 (8, 9),
 (9, 0)]

In [154]:
edges = list(graph.edges(data=True))

In [155]:
graph[5][6]['pts']

array([[347,  14],
       [349,  14],
       [350,  14],
       [351,  14],
       [352,  14],
       [353,  14],
       [354,  14],
       [355,  14],
       [356,  14],
       [357,  14],
       [358,  14],
       [359,  14],
       [360,  14],
       [361,  14],
       [362,  14],
       [363,  14],
       [364,  14],
       [365,  14],
       [366,  14],
       [367,  14],
       [368,  14],
       [369,  15],
       [370,  15],
       [371,  15],
       [372,  15],
       [373,  15],
       [374,  15],
       [375,  16],
       [376,  16],
       [377,  16],
       [378,  16],
       [379,  16],
       [380,  16],
       [381,  16],
       [382,  16],
       [383,  16],
       [384,  16],
       [385,  17],
       [386,  17],
       [387,  17],
       [388,  17],
       [389,  17],
       [390,  17],
       [391,  17],
       [392,  17],
       [393,  17],
       [394,  17],
       [395,  17],
       [396,  17],
       [397,  18],
       [398,  18],
       [399,  18],
       [400,

In [156]:
ordered_nodes

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [157]:
all_nodes = []

for i in range(len(ordered_nodes)):
    n1 = ordered_nodes[i]
    n2 = ordered_nodes[(i+1)%len(ordered_nodes)]
    all_nodes.extend(graph[n1][n2]['pts'])

all_nodes = np.array(all_nodes)
all_nodes, len(all_nodes)

(array([[60, 84],
        [57, 87],
        [56, 88],
        ...,
        [64, 82],
        [65, 82],
        [67, 80]], dtype=int16),
 838)

### Visualization

In [158]:
path_test = np.zeros_like(map_)
path_test = cv2.cvtColor(path_test, cv2.COLOR_GRAY2BGR)

for edge in edges:
    path_test = cv2.circle(path_test, (edge[2]["pts"][0][1], edge[2]["pts"][0][0]), 1, (0, 0, 255), 1)
    path_test = cv2.circle(path_test, (edge[2]["pts"][-1][1], edge[2]["pts"][-1][0]), 1, (0, 0, 255), 1)
    path_test = cv2.putText(path_test, f"{edge[0]}", (edge[2]["pts"][0][1], edge[2]["pts"][0][0]), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)
    if (edge[0] == 7 or edge[1] == 7):
        print(f"Point 7, start: ({edge[2]['pts'][0][1]}, {edge[2]['pts'][0][0]}), end: ({edge[2]['pts'][-1][1]}, {edge[2]['pts'][-1][0]})")
    # path_test = cv2.putText(path_test, f"{edge[1]}", (edge[2]["pts"][-1][1], edge[2]["pts"][-1][0]), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)
    # Set the path color to white
    for coor in edge[2]["pts"]:
        path_test[coor[0], coor[1]] = (255, 255, 255)

cv2.imshow("test", path_test)
cv2.waitKey()
cv2.destroyAllWindows()

Point 7, start: (70, 334), end: (19, 342)
Point 7, start: (19, 342), end: (14, 347)


### Test Error

In [159]:
half_total = []

for i in range(13, len(ordered_nodes)):
    n1 = ordered_nodes[i]
    n2 = ordered_nodes[(i+1)%len(ordered_nodes)]
    half_total.extend(graph[n1][n2]['pts'])

half_total = np.array(half_total)
half_total = half_total[np.argsort(half_total[:, 0])[::-1]]
half_total, len(half_total)

IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed

In [136]:
# Test and visualized if the node is correctly structured
path_test = np.zeros((600, 600, 3))
print(path_test.shape)
#path_test = cv2.cvtColor(path_test, cv2.COLOR_GRAY2BGR)
for i in range(half_total.shape[0]):
    path_test[half_total[i][0] + 100, half_total[i][1] + 100] = (255, 255, 255)

cv2.imshow("test", path_test)
cv2.waitKey()
cv2.destroyAllWindows()

(600, 600, 3)


In [75]:
# Test and visualized if the node is correctly structured
path_test = np.zeros((1200, 1200, 3))
#path_test = cv2.cvtColor(path_test, cv2.COLOR_GRAY2BGR)
for i in range(half_total.shape[0]):
    n_1 = half_total[i]
    n_2 = half_total[(i+1)%half_total.shape[0]]
    path_test = cv2.circle(path_test, (10+200, 411+200), 1, (0, 0, 255), 2)
    path_test = cv2.circle(path_test, (59+200, 120+200), 1, (0, 0, 255), 2)
    path_test = cv2.putText(path_test, "A", (59+200, 120+200), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)
    path_test = cv2.arrowedLine(path_test, (n_1[1]+200, n_1[0]+200), (n_2[1]+200, n_2[0]+200), (0, 255, 0), 1, cv2.LINE_AA, tipLength=0.5)
    cv2.imshow("test", path_test)
    c = cv2.waitKey(1)
    if c == 27:
        break
cv2.destroyAllWindows()

### Test Error 2

In [76]:
another_half_total = []

for i in range(8, 13):
    n1 = ordered_nodes[i]
    n2 = ordered_nodes[(i+1)%len(ordered_nodes)]
    another_half_total.extend(graph[n1][n2]['pts'])

another_half_total = np.array(another_half_total)
another_half_total = another_half_total[np.argsort(another_half_total[:, 1])[::-1]]
len(another_half_total)

81

In [77]:
# Test and visualized if the node is correctly structured
path_test = np.zeros((1200, 1200, 3))
#path_test = cv2.cvtColor(path_test, cv2.COLOR_GRAY2BGR)
for i in range(another_half_total.shape[0]):
    n_1 = another_half_total[i]
    n_2 = another_half_total[(i+1)%another_half_total.shape[0]]
    path_test = cv2.circle(path_test, (10+200, 411+200), 1, (0, 0, 255), 2)
    path_test = cv2.circle(path_test, (59+200, 120+200), 1, (0, 0, 255), 2)
    path_test = cv2.putText(path_test, "A", (59+200, 120+200), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)
    path_test = cv2.arrowedLine(path_test, (n_1[1]+200, n_1[0]+200), (n_2[1]+200, n_2[0]+200), (0, 255, 0), 1, cv2.LINE_AA, tipLength=0.5)
    cv2.imshow("test", path_test)
    c = cv2.waitKey(1)
    if c == 27:
        break
cv2.destroyAllWindows()

## Creating The Path

In [78]:
all_nodes = []

for i in range(8):
    n1 = ordered_nodes[i]
    n2 = ordered_nodes[(i+1)%len(ordered_nodes)]
    all_nodes.extend(graph[n1][n2]['pts'])

all_nodes = np.array(all_nodes)
all_nodes, len(all_nodes)

(array([[ 60,  69],
        [ 58,  71],
        [ 58,  72],
        [ 58,  73],
        [ 58,  74],
        [ 59,  75],
        [ 59,  76],
        [ 60,  77],
        [ 62,  79],
        [ 62,  79],
        [ 63,  81],
        [ 64,  82],
        [ 64,  82],
        [ 64,  83],
        [ 64,  82],
        [ 64,  82],
        [ 66,  84],
        [ 67,  84],
        [ 68,  85],
        [ 69,  85],
        [ 70,  86],
        [ 71,  86],
        [ 72,  87],
        [ 73,  87],
        [ 74,  88],
        [ 75,  88],
        [ 76,  89],
        [ 78,  90],
        [ 78,  90],
        [ 81,  92],
        [ 82,  92],
        [ 83,  93],
        [ 84,  94],
        [ 87,  95],
        [ 87,  95],
        [ 90,  96],
        [ 91,  97],
        [ 92,  97],
        [ 93,  98],
        [ 94,  98],
        [ 95,  98],
        [ 96,  99],
        [ 97,  99],
        [ 98,  99],
        [ 99, 100],
        [100, 100],
        [101, 101],
        [102, 101],
        [103, 101],
        [104, 102],


In [79]:
len(half_total)

401

In [80]:
all_nodes = np.concatenate([all_nodes, another_half_total], axis=0)

In [81]:
all_nodes = np.concatenate([all_nodes, half_total], axis=0)
all_nodes, len(all_nodes)

(array([[60, 69],
        [58, 71],
        [58, 72],
        ...,
        [63, 67],
        [62, 68],
        [60, 69]], dtype=int16),
 825)

In [82]:
# Test and visualized if the node is correctly structured
path_test = np.zeros((600, 600, 3))
print(path_test.shape)
#path_test = cv2.cvtColor(path_test, cv2.COLOR_GRAY2BGR)
for i in range(all_nodes.shape[0]):
    path_test[all_nodes[i][0] + 100, all_nodes[i][1] + 100] = (255, 255, 255)

cv2.imshow("test", path_test)
cv2.waitKey()
cv2.destroyAllWindows()

(600, 600, 3)


In [83]:
cv2.imshow("real_map", map_)
cv2.waitKey()
cv2.destroyAllWindows()

In [84]:
# Test and visualized if the node is correctly structured
path_test = np.zeros((1200, 1200, 3))
#path_test = cv2.cvtColor(path_test, cv2.COLOR_GRAY2BGR)
for i in range(all_nodes.shape[0]):
    n_1 = all_nodes[i]
    n_2 = all_nodes[(i+1)%all_nodes.shape[0]]
    path_test = cv2.arrowedLine(path_test, (n_1[1]+200, n_1[0]+200), (n_2[1]+200, n_2[0]+200), (0, 255, 0), 1, cv2.LINE_AA, tipLength=0.5)
    cv2.imshow("test", path_test)
    c = cv2.waitKey(1)
    if c == 27:
        break
cv2.destroyAllWindows()

In [85]:
import yaml
with open('map_1764936172.yaml', 'r') as f:
    data = yaml.load(f, Loader=yaml.SafeLoader)
print(data)

_res = data["resolution"]
_origin = data["origin"]

{'image': 'map_1764936172.pgm', 'mode': 'trinary', 'resolution': 0.05, 'origin': [-4.62, -10.7, 0], 'negate': 0, 'occupied_thresh': 0.65, 'free_thresh': 0.25}


In [88]:
# Convert the pixel coordinate to map coordinate
_world_coord = []
for i in range(all_nodes.shape[0]):
    n_1 = all_nodes[i]
    n_2 = all_nodes[(i+1)%all_nodes.shape[0]]
    _coor_x_1 = (_res * n_1[1]) + _origin[0]
    _coor_y_1 = (_res * (map_.shape[0] - n_1[0])) + _origin[1]
    _coor_x_2 = (_res * n_2[1]) + _origin[0]
    _coor_y_2 = (_res * (map_.shape[0] - n_2[0])) + _origin[1]
    _yaw = math.atan2(_coor_y_2-_coor_y_1, _coor_x_2-_coor_x_1)
    _world_coord.append((_coor_x_1 + 0.5, _coor_y_1, 0.1, _yaw)) # [x, y, v, yaw]
print(_world_coord)

[(-0.6699999999999999, 10.000000000000004, 0.1, 0.7853981633974372), (-0.5699999999999998, 10.100000000000001, 0.1, 0.0), (-0.52, 10.100000000000001, 0.1, 0.0), (-0.46999999999999975, 10.100000000000001, 0.1, 0.0), (-0.41999999999999993, 10.100000000000001, 0.1, -0.7853981633974572), (-0.3700000000000001, 10.05, 0.1, 0.0), (-0.31999999999999984, 10.05, 0.1, -0.7853981633974216), (-0.27, 10.000000000000004, 0.1, -0.7853981633974549), (-0.16999999999999993, 9.900000000000002, 0.1, 0.0), (-0.16999999999999993, 9.900000000000002, 0.1, -0.4636476090008132), (-0.07000000000000028, 9.850000000000001, 0.1, -0.7853981633974483), (-0.019999999999999574, 9.8, 0.1, 0.0), (-0.019999999999999574, 9.8, 0.1, 0.0), (0.03000000000000025, 9.8, 0.1, 3.141592653589793), (-0.019999999999999574, 9.8, 0.1, 0.0), (-0.019999999999999574, 9.8, 0.1, -0.7853981633974394), (0.08000000000000007, 9.700000000000003, 0.1, -1.5707963267948966), (0.08000000000000007, 9.650000000000002, 0.1, -0.7853981633974572), (0.12999

In [89]:
import csv
with open('path.csv', "w", newline="") as f:
    writer = csv.writer(f)
    # writer.writerow(["x", "y", "velocity", "yaw"])
    writer.writerows(_world_coord)
